# Advanced Problems with Solutions: Python 3.8 Assignment Expressions (`:=`)

Practice advanced uses of assignment expressions: comprehensions, stream processing, regex matching, scoping, syntax restrictions, and readability trade-offs.


## Problem 1 — Avoid repeated expensive validation in a comprehension

`parse_record(raw)` is expensive. Write `select_strong_records(records)` using a list comprehension and `:=`.

Requirements:
- Call `parse_record` exactly once per record.
- Keep only valid records.
- Keep only records with `score >= 80`.
- Return parsed dictionaries.


In [1]:
parse_calls = 0

def parse_record(raw):
    global parse_calls
    parse_calls += 1
    try:
        name, score_text = raw.split(':')
        score = int(score_text)
    except ValueError:
        return None
    name = name.strip().title()
    if not name:
        return None
    return {'name': name, 'score': score}

records = ['alice:91', 'bad-row', 'bob:79', 'carol:88', ':100', 'dave:x']

def select_strong_records(records):
    return [
        parsed
        for raw in records
        if (parsed := parse_record(raw)) is not None and parsed['score'] >= 80
    ]

parse_calls = 0
answer = select_strong_records(records)

assert answer == [{'name': 'Alice', 'score': 91}, {'name': 'Carol', 'score': 88}]
assert parse_calls == len(records)

answer, parse_calls


([{'name': 'Alice', 'score': 91}, {'name': 'Carol', 'score': 88}], 6)

### Solution

The assignment expression belongs in the `if` clause so the parsed value is computed once, checked, and then reused as the value emitted by the comprehension.


## Problem 2 — Comprehension evaluation order

Predict why `good` works but `bad` fails. Then fix `bad` while still using `:=`.


In [2]:
def transform(x):
    print(f'transform({x})')
    return x * x - 3

good = [
    y
    for x in range(5)
    if (y := transform(x)) > 0
]

print('good:', good)

try:
    bad = [
        (z := transform(x))
        for x in range(5)
        if z > 0
    ]
except NameError as exc:
    print('bad failed:', exc)

fixed = [
    z
    for x in range(5)
    if (z := transform(x)) > 0
]

assert good == fixed == [1, 6, 13]
fixed


transform(0)
transform(1)
transform(2)
transform(3)
transform(4)
good: [1, 6, 13]
bad failed: name 'z' is not defined
transform(0)
transform(1)
transform(2)
transform(3)
transform(4)


[1, 6, 13]

### Solution

In a comprehension with an `if`, Python evaluates the loop variable, then the `if` condition, then the element expression. Therefore the assignment must happen in the `if` clause if the `if` clause needs the assigned value.


## Problem 3 — Stream processing until a sentinel

Write `read_until_stop(stream)` using `while` and `:=`.

Requirements:
- Read one line per iteration.
- Stop on end-of-stream.
- Stop when the stripped line is `'STOP'`.
- Return all previous stripped lines.


In [3]:
from io import StringIO

def read_until_stop(stream):
    result = []
    while (line := stream.readline()):
        clean = line.strip()
        if clean == 'STOP':
            break
        result.append(clean)
    return result

assert read_until_stop(StringIO('alpha\n beta \nSTOP\ngamma\n')) == ['alpha', 'beta']
assert read_until_stop(StringIO('one\ntwo\n')) == ['one', 'two']

read_until_stop(StringIO('x\ny\nSTOP\nz\n'))


['x', 'y']

### Solution

`while (line := stream.readline()):` both consumes the next line and checks whether the stream still has data.


## Problem 4 — Regex extraction without repeated matching

Extract only messages where `LEVEL == 'ERROR'` and the component is `'api'`.

Requirements:
- Use one regex match attempt per line.
- Use `:=`.
- Return stripped messages.


In [4]:
import re

LOG_PATTERN = re.compile(r'^(?P<level>[A-Z]+)\s+\[(?P<component>[^\]]+)\]\s+(?P<message>.*)$')

lines = [
    'INFO [api] started',
    'ERROR [worker] timeout',
    'ERROR [api] invalid token',
    'DEBUG [api] payload received',
    'not a valid log line',
    'ERROR [api]   database unavailable   ',
]

def api_errors(lines):
    return [
        match.group('message').strip()
        for line in lines
        if (match := LOG_PATTERN.match(line))
        and match.group('level') == 'ERROR'
        and match.group('component') == 'api'
    ]

assert api_errors(lines) == ['invalid token', 'database unavailable']
api_errors(lines)


['invalid token', 'database unavailable']

### Solution

The regex match object is bound once in the condition and reused for filtering and extracting the final message.


## Problem 5 — Scope surprise after a comprehension

Assignment-expression targets inside comprehensions bind in the surrounding scope. Explain why `item` is not available afterward, but `last_square` is.


In [5]:
for name in ['item', 'last_square']:
    try:
        del globals()[name]
    except KeyError:
        pass

values = [
    last_square
    for item in range(8)
    if (last_square := item * item) % 2 == 0
]

print('values:', values)

try:
    print('item:', item)
except NameError as exc:
    print('item is not available:', exc)

print('last_square:', last_square)

assert values == [0, 4, 16, 36]
assert last_square == 49


values: [0, 4, 16, 36]
item is not available: name 'item' is not defined
last_square: 49


### Solution

The comprehension loop variable `item` does not leak, but the walrus target `last_square` is bound in the surrounding scope. It is assigned on every iteration before the filter is applied, so after `item == 7`, `last_square` is `49` even though `49` is not included in `values`.


## Problem 6 — Syntax restrictions and parentheses

For each snippet, predict whether it is valid. Then run the cell and study the valid equivalents.


In [6]:
snippets = {
    'plain_statement': 'x := 10',
    'parenthesized_expression_statement': '(x := 10)',
    'if_condition': 'if x := 10:\n    pass',
    'if_condition_parenthesized': 'if (x := 10):\n    pass',
    'attribute_target': '(obj.attr := 10)',
    'subscript_target': '(items[0] := 10)',
}

for name, source in snippets.items():
    try:
        compile(source, filename=f'<{name}>', mode='exec')
        print(f'{name:35} VALID')
    except SyntaxError as exc:
        print(f'{name:35} INVALID -> {exc.msg}')

# Valid equivalents
x = 10

if (x := 10):
    pass

class Box:
    pass

obj = Box()
obj.attr = 10

items = [0]
items[0] = 10

x, obj.attr, items[0]


plain_statement                     INVALID -> invalid syntax
parenthesized_expression_statement  VALID
if_condition                        VALID
if_condition_parenthesized          VALID
attribute_target                    INVALID -> cannot use assignment expressions with attribute
subscript_target                    INVALID -> cannot use assignment expressions with subscript


(10, 10, 10)

### Solution

`x := 10` is not a valid standalone statement. Assignment-expression targets must be simple names, so `obj.attr := 10` and `items[0] := 10` are invalid. Use ordinary assignment for those cases.


## Problem 7 — Chunked reader with exact call count

Write `read_chunks(reader, size)` using `while` and `:=`.

Requirements:
- Return all non-empty chunks.
- Call `.read(size)` once per returned chunk, plus once for the final empty sentinel.
- Raise `ValueError` for `size <= 0`.


In [7]:
class CountingBytesReader:
    def __init__(self, data):
        self._data = data
        self.calls = 0

    def read(self, size):
        self.calls += 1
        chunk, self._data = self._data[:size], self._data[size:]
        return chunk

def read_chunks(reader, size):
    if size <= 0:
        raise ValueError('size must be positive')

    chunks = []
    while chunk := reader.read(size):
        chunks.append(chunk)
    return chunks

reader = CountingBytesReader(b'abcdefghij')
chunks = read_chunks(reader, 4)

assert chunks == [b'abcd', b'efgh', b'ij']
assert reader.calls == 4

try:
    read_chunks(CountingBytesReader(b'abc'), 0)
except ValueError:
    pass
else:
    raise AssertionError('Expected ValueError')

chunks, reader.calls


([b'abcd', b'efgh', b'ij'], 4)

### Solution

`while chunk := reader.read(size):` avoids the common duplicate-read pattern. The final empty read stops the loop.


## Problem 8 — Refactor an over-clever use of `:=`

The first function works, but it is too dense. Write a clearer version. You may still use `:=`, but only where it improves readability.


In [8]:
def original_parse_assignments(text):
    return {
        key: int(value)
        for line in text.splitlines()
        if (clean := line.strip())
        and not clean.startswith('#')
        and '=' in clean
        and (key := clean.split('=', 1)[0].strip())
        and (value := clean.split('=', 1)[1].strip()).isdigit()
    }

sample = '''
# config
retries = 3
timeout = 30
bad-line
empty =
pi = 3.14
workers = 8
'''

assert original_parse_assignments(sample) == {'retries': 3, 'timeout': 30, 'workers': 8}

def clearer_parse_assignments(text):
    parsed = {}

    for line in text.splitlines():
        if not (clean := line.strip()):
            continue

        if clean.startswith('#') or '=' not in clean:
            continue

        key_text, value_text = clean.split('=', 1)
        key = key_text.strip()
        value = value_text.strip()

        if not key or not value.isdigit():
            continue

        parsed[key] = int(value)

    return parsed

assert clearer_parse_assignments(sample) == original_parse_assignments(sample)
clearer_parse_assignments(sample)


{'retries': 3, 'timeout': 30, 'workers': 8}

### Solution

The original version overuses `:=` and repeats `clean.split('=', 1)`. The clearer version keeps one useful walrus expression for trimming and checking an empty line, then uses ordinary statements for the rest.


## Final checklist

Use assignment expressions when they avoid repeated expensive work, simplify stream-reading loops, or keep a parsed/matched/generated value close to the condition that validates it.

Avoid assignment expressions when they create dense business logic, depend on surprising evaluation order, leave confusing names in the surrounding scope, or try to replace a simple `=` statement.
